In [2]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

# dataset
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1,1), dtype=torch.float32)

# model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 6)
        self.fc2 = nn.Linear(6, 1)

    def forward(self, x):
        z1 = self.fc1(x)          # W1·x + b1
        a1 = torch.tanh(z1)       # try relu/tanh
        z2 = self.fc2(a1)         # W2·a1 + b2
        return torch.sigmoid(z2)  # output

model = Net()

#### Frame Generator (Decision Boundary → Image)

In [6]:
def get_frame(model, X, y, epoch):
    import numpy as np
    import matplotlib.pyplot as plt

    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

    with torch.no_grad():
        preds = model(grid).reshape(xx.shape)

    fig, ax = plt.subplots()
    ax.contourf(xx, yy, preds.numpy(), alpha=0.6)
    ax.scatter(X[:,0], X[:,1], c=y.squeeze(), edgecolors='k')
    ax.set_title(f"Epoch {epoch}")

    fig.canvas.draw()

    # ✅ FIXED PART
    image = np.asarray(fig.canvas.buffer_rgba())
    image = image[:, :, :3]

    plt.close(fig)

    return image

### Train + Capture Frames

In [7]:
import imageio

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

frames = []

for epoch in range(201):
    # forward
    y_pred = model(X)
    loss = loss_fn(y_pred, y)

    # backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # capture every few steps
    if epoch % 5 == 0:
        frame = get_frame(model, X, y, epoch)
        frames.append(frame)

In [8]:
imageio.mimsave("training.gif", frames, fps=5)

In [9]:
def get_frame_with_loss(model, X, y, epoch, losses):
    import numpy as np
    import matplotlib.pyplot as plt

    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 150),
        np.linspace(y_min, y_max, 150)
    )

    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

    with torch.no_grad():
        preds = model(grid).reshape(xx.shape)

    # 🔥 Create 2 subplots
    fig, axs = plt.subplots(1, 2, figsize=(10,4))

    # LEFT: Decision Boundary
    axs[0].contourf(xx, yy, preds.numpy(), alpha=0.6)
    axs[0].scatter(X[:,0], X[:,1], c=y.squeeze(), edgecolors='k')
    axs[0].set_title(f"Decision Boundary (Epoch {epoch})")

    # RIGHT: Loss Curve
    axs[1].plot(losses, marker='o')
    axs[1].set_title("Loss Curve")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Loss")

    plt.tight_layout()
    fig.canvas.draw()

    # Convert to image
    image = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)

    return image

In [10]:
import imageio

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

frames = []
losses = []

for epoch in range(201):
    # Forward
    y_pred = model(X)
    loss = loss_fn(y_pred, y)

    # Store loss
    losses.append(loss.item())

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Capture frame
    if epoch % 5 == 0:
        frame = get_frame_with_loss(model, X, y, epoch, losses)
        frames.append(frame)

In [11]:
imageio.mimsave("training_with_loss.gif", frames, fps=5)